# Phase 3 - Hoang Grounded Output Research

Notebook nay tap trung vao phan cuoi cua Vinh Hoang:
1. Load phase 1 artifact
2. Consume San phase 2 contract (or local mock/sample)
3. Generate grounded answer and export `phase3_hoang_grounded_answer_output.jsonl`


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'aviation_rag').exists() and (ROOT.parent / 'aviation_rag').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from aviation_rag.config import Settings
from aviation_rag.io_utils import read_jsonl, write_jsonl
from aviation_rag.phase2_san_contract_adapter import Phase2SanContractAdapter
from aviation_rag.phase3_hoang_grounded_qa import Phase3HoangGroundedQA
from aviation_rag.schemas import InputAgentOutput

settings = Settings()
print('Phase 1 artifact:', settings.phase1_output_path)
print('Phase 2 artifact:', settings.phase2_output_path)
print('Phase 2 sample artifact:', settings.phase2_sample_output_path)
print('Phase 3 artifact:', settings.phase3_output_path)


## Load Phase 1 Artifact


In [ ]:
phase1_rows = read_jsonl(settings.phase1_output_path)
if not phase1_rows:
    raise ValueError(f'No rows found in {settings.phase1_output_path}. Run phase 1 notebook first.')
phase1_rows[0]


## Resolve San Phase 2 Contract

Neu San chua co output that, adapter se dung sample/mock de demo full flow.


In [ ]:
phase2_adapter = Phase2SanContractAdapter(settings)
phase2_rows = []
for row in phase1_rows:
    phase1_output = InputAgentOutput.model_validate(row)
    phase2_output = phase2_adapter.resolve_output(phase1_output)
    phase2_rows.append(phase2_output)

write_jsonl(settings.phase2_output_path, phase2_rows)
[
    {
        'query_id': row.query_id,
        'predicted_intent': row.predicted_intent,
        'adapter_mode': row.retrieval_diagnostics.get('adapter_mode'),
        'topk_docs': len(row.topk_docs),
    }
    for row in phase2_rows
]


## Generate Hoang Phase 3 Grounded Output


In [ ]:
phase3 = Phase3HoangGroundedQA(settings)
phase1_lookup = {row['query_id']: row for row in phase1_rows}
phase3_rows = []
for phase2_output in phase2_rows:
    query_raw = phase1_lookup[phase2_output.query_id]['query_raw']
    phase3_rows.append(
        phase3.generate(
            question=query_raw,
            middle_output=phase2_output,
            allow_fallback=True,
        )
    )

[
    {
        'query_id': row.query_id,
        'answer_preview': row.answer[:160],
        'hallucination_risk': row.hallucination_risk,
    }
    for row in phase3_rows
]


## Export Phase 3 Artifact


In [ ]:
write_jsonl(settings.phase3_output_path, phase3_rows)
required = ['query_id', 'answer', 'citations', 'hallucination_risk', 'grounding_report', 'created_at']
for row in phase3_rows:
    payload = row.model_dump(mode='json')
    missing = [key for key in required if key not in payload]
    if missing:
        raise ValueError(f'Missing keys in {payload.get("query_id")}: {missing}')
print(f'Wrote {len(phase3_rows)} rows to {settings.phase3_output_path}')
print('Phase 3 contract validation passed.')
